In [1]:
import os
from google.colab import drive

drive.mount('/content/drive')
os.chdir('/content')

# remove old folders
!rm -rf /content/GRU4Rec_PyTorch_Official
!rm -rf /content/safe_data

!git clone https://github.com/hidasib/GRU4Rec_PyTorch_Official.git
print("\n[INFO] Extracting dataset from Drive\n")
!unzip -q -o "/content/drive/MyDrive/TM2V3/archive (2).zip" -d "/content/GRU4Rec_PyTorch_Official/data/"
print("[SUCCESS] Code and data are set up correctly!")

Mounted at /content/drive
Cloning into 'GRU4Rec_PyTorch_Official'...
remote: Enumerating objects: 75, done.
remote: Counting objects: 100% (27/27), done.
remote: Compressing objects: 100% (12/12), done.
remote: Total 75 (delta 20), reused 15 (delta 15), pack-reused 48 (from 1)
Receiving objects: 100% (75/75), 362.75 KiB | 18.14 MiB/s, done.
Resolving deltas: 100% (35/35), done.

[INFO] Extracting dataset from Drive

[SUCCESS] Code and data are set up correctly!


In [2]:
import pandas as pd
import numpy as np
import gc
import os

os.chdir('/content/GRU4Rec_PyTorch_Official')
input_file = 'data/yoochoose-clicks.dat'
output_file = 'data/yoochoose_processed.csv'

print("[INFO] Loading and processing raw data")
df = pd.read_csv(input_file, sep=',', header=None, dtype={0: np.int32, 2: np.int32}, usecols=[0, 1, 2])
df.columns = ['SessionId', 'Timestamp', 'ItemId']

# filter short sessions and non-active users
item_supports = df.groupby('ItemId').size()
df = df[np.in1d(df.ItemId, item_supports[item_supports >= 5].index)]
session_lengths = df.groupby('SessionId').size()
df = df[np.in1d(df.SessionId, session_lengths[session_lengths >= 2].index)]

# memory free
del item_supports, session_lengths
gc.collect()

# convert timestamps to Unix Epoch for faster sorting
df['Time'] = pd.to_datetime(df['Timestamp'], format='%Y-%m-%dT%H:%M:%S.%fZ').astype(np.int64) // 10**9
df.drop(columns=['Timestamp'], inplace=True)
df = df.sort_values(['SessionId', 'Time'])
gc.collect()

# save
df.to_csv(output_file, sep='\t', index=False)
print("[SUCCESS] Data is processed and ready for training!")

[INFO] Loading and processing raw data


/tmp/ipykernel_1665/2341176906.py:16: DeprecationWarning: `in1d` is deprecated. Use `np.isin` instead.
  df = df[np.in1d(df.ItemId, item_supports[item_supports >= 5].index)]
/tmp/ipykernel_1665/2341176906.py:18: DeprecationWarning: `in1d` is deprecated. Use `np.isin` instead.
  df = df[np.in1d(df.SessionId, session_lengths[session_lengths >= 2].index)]


[SUCCESS] Data is processed and ready for training!


In [3]:
# patch 1
file_path = '/content/GRU4Rec_PyTorch_Official/gru4rec_pytorch.py'

with open(file_path, 'r') as file:
    code = file.read()

# force the tensors to explicitly use float32 to match the model weights
code = code.replace("torch.tensor(np.vstack(m)", "torch.tensor(np.vstack(m), dtype=torch.float32")
code = code.replace("torch.tensor(np.vstack(m2)", "torch.tensor(np.vstack(m2), dtype=torch.float32")
code = code.replace("torch.tensor(np.hstack(b)", "torch.tensor(np.hstack(b), dtype=torch.float32")
code = code.replace("torch.tensor(np.hstack(b2)", "torch.tensor(np.hstack(b2), dtype=torch.float32")

with open(file_path, 'w') as file:
    file.write(code)

print("[SUCCESS] Repository bug fixed!")

[SUCCESS] Repository bug fixed!


In [4]:
# patch 2
file_path = '/content/GRU4Rec_PyTorch_Official/gru4rec_pytorch.py'

with open(file_path, 'r') as file:
    code = file.read()

# patch last bug
code = code.replace(
    "torch.tensor(self._init_numpy_weights((self.n_items, self.layers[-1])), device=self.Wy.weight.device)",
    "torch.tensor(self._init_numpy_weights((self.n_items, self.layers[-1])), dtype=torch.float32, device=self.Wy.weight.device)"
)

with open(file_path, 'w') as file:
    file.write(code)

print("[SUCCESS] Done!")

[SUCCESS] Done!


In [5]:
# run training script
!python run.py data/yoochoose_processed.csv \
  -ps "loss=bpr-max,batch_size=512,learning_rate=0.05,n_epochs=5,layers=100"

Creating GRU4Rec model on device "cuda:0"
SET   loss            TO   bpr-max   (type: <class 'str'>)
SET   batch_size      TO   512       (type: <class 'int'>)
SET   learning_rate   TO   0.05      (type: <class 'float'>)
SET   n_epochs        TO   5         (type: <class 'int'>)
SET   layers          TO   [100]     (type: <class 'list'>)
Loading training data...
Loading data from TAB separated file: data/yoochoose_processed.csv
Started training
The dataframe is already sorted by SessionId, Time
Created sample store with 4882 batches of samples (type=GPU)
Epoch1 --> loss: 0.370040 	(213.50s) 	[217.33 mb/s | 111271 e/s]
Epoch2 --> loss: 0.360741 	(212.32s) 	[218.54 mb/s | 111892 e/s]
Epoch3 --> loss: 0.359173 	(212.38s) 	[218.48 mb/s | 111860 e/s]
Epoch4 --> loss: 0.358611 	(211.30s) 	[219.59 mb/s | 112432 e/s]
Epoch5 --> loss: 0.358360 	(209.72s) 	[221.25 mb/s | 113279 e/s]
Total training time: 1082.17s


In [6]:
import pandas as pd
import gru4rec_pytorch
import evaluation

# load dataset for performance results
print("[INFO] Loading dataset...")
data = pd.read_csv('data/yoochoose_processed.csv', sep='\t')

# data split
print("[INFO] Splitting data into Train and Test sets...")
split_point = int(len(data) * 0.9)
train_data = data.iloc[:split_point]
test_data = data.iloc[split_point:]

# gruRec init - one epochs for quick checkup
print("[INFO] Initializing model...")
model = gru4rec_pytorch.GRU4Rec(
    loss='bpr-max',
    batch_size=512,
    learning_rate=0.05,
    n_epochs=1,
    layers=[100]
)

print("[INFO] Starting training...")
model.fit(train_data)

print("[INFO] Evaluating the model using 'batch_eval'...")
eval_results = evaluation.batch_eval(model, test_data, [20])
recall_dict, mrr_dict = eval_results
recall_20 = recall_dict[20]
mrr_20 = mrr_dict[20]

print(f"\nFINAL RESULTS:")
print(f"Recall@20: {recall_20:.4f}")
print(f"MRR@20:    {mrr_20:.4f}")

[INFO] Loading dataset...
[INFO] Splitting data into Train and Test sets...
[INFO] Initializing model...
[INFO] Starting training...
The dataframe is already sorted by SessionId, Time
Created sample store with 4882 batches of samples (type=GPU)
Epoch1 --> loss: 0.370933 	(189.58s) 	[220.12 mb/s | 112702 e/s]
[INFO] Evaluating the model using 'batch_eval'...
Using existing item ID map
The dataframe is already sorted by SessionId, Time

FINAL RESULTS:
Recall@20: 0.6733
MRR@20:    0.2997
